# EEG STFT Analysis Suite

This notebook implements a modular STFT pipeline with custom tapering (windowing) functions,
sensitivity analyses, and spectrogram visualizations.

**Report Structure**
1. Theoretical Background
2. Methodology
3. Code Explanation
4. Results and Discussion

## 1. Theoretical Background

The short-time Fourier transform (STFT) analyzes nonstationary signals by applying a window
to short signal segments and transforming each segment into the frequency domain.

STFT definition:
$$
X(m, k) = \sum_{n=0}^{N-1} x[n + mR] \, w[n] \, e^{-j 2\pi k n / N}
$$

Power (magnitude squared):
$$
P(m, k) = |X(m, k)|^2
$$

where $N$ is the window length and $R$ is the hop size.

In [31]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve().parents[1]
print(f"Project root directory: {PROJECT_ROOT}")
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print(f"Added {SRC_DIR} to sys.path")
from eeg_io import load_first_channel_edf
from signal_generation import linear_chirp
from stft import compute_window_length, stft_power
from windows import rectangular, hann, hamming, blackman, gaussian
from analysis_utils import overlap_sweep
from plotting import plot_spectrogram

FIG_DIR = PROJECT_ROOT / 'notebooks' / 'week2' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

Project root directory: /home/plutonium/Documents/Biosignals/project
Added /home/plutonium/Documents/Biosignals/project/src to sys.path


## 2. Methodology

### 2.1 Signal and Parameters
- Chirp signal: 10 s, 200 Hz sampling, 5 Hz to 100 Hz.
- Window length $N$ is set to capture at least 3 cycles at 4 Hz.

### 2.2 STFT Steps
1. Segment the signal into length-$N$ frames.
2. Multiply each frame by a window.
3. Compute FFT and keep 4--100 Hz.
4. Compute power spectrum $|X|^2$.
5. Map each frame to its center time.

In [32]:
# Chirp configuration
fs = 200.0
duration = 10.0
f0 = 5.0
f1 = 100.0

t, x = linear_chirp(duration, fs, f0, f1)

# Window length for at least 3 cycles at 4 Hz
f_min = 4.0
nperseg = compute_window_length(fs, f_min, cycles=3)
nperseg

150

In [33]:
# Plot the original chirp waveform
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(t, x, color='tab:blue', linewidth=1.0)
ax.set_title(f'Chirp waveform (fs={fs:.1f} Hz, duration={duration:.1f} s, f0={f0:.1f} Hz, f1={f1:.1f} Hz)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.grid(alpha=0.3)
fig.tight_layout()
save_path = FIG_DIR / 'chirp_waveform.png'
fig.savefig(save_path, dpi=180)
plt.close(fig)
print(f'Saved: {save_path}')

Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_waveform.png


## 3. Code Explanation

The core logic is implemented in `project/src` modules. The STFT computation is explicit:
segmentation, tapering, FFT, frequency filtering, and power mapping.

In [34]:
# Overlap sweep from 0% to 95% in 5% steps
overlap_fracs = [i / 100 for i in range(0, 100, 5) if i < 95] + [0.95]
overlap_results = overlap_sweep(
    x,
    fs,
    window_name='hann',
    nperseg=nperseg,
    overlap_fracs=overlap_fracs,
    fmin=4.0,
    fmax=100.0,
)

# Plot a subset of overlap settings for readability
for frac in [0.0, 0.5, 0.9, 0.95]:
    freqs, times, power = overlap_results[frac]
    title = (
        f'Chirp STFT (window=Hann, overlap={int(frac*100)}%, '
        f'N={nperseg}, fs={fs:.1f} Hz, fmin=4 Hz, fmax=100 Hz)'
    )
    save_path = FIG_DIR / f'chirp_hann_overlap_{int(frac*100)}.png'
    plot_spectrogram(freqs, times, power, title=title, save_path=save_path)
    print(f'Saved: {save_path}')

Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_hann_overlap_0.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_hann_overlap_50.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_hann_overlap_90.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_hann_overlap_95.png


## 4. Results and Discussion

### 4.1 Hamming Sensitivity
We vary $a_1$ by +/-20% with $a_0$ fixed, then vary $a_0$ by +/-20% with $a_1$ fixed.

In [35]:
# Hamming sensitivity
hamming_base = {'a0': 0.54, 'a1': 0.46}
hamming_variations = []
for scale in [0.8, 1.0, 1.2]:
    hamming_variations.append((f'a1_{scale:.2f}', {'a0': 0.54, 'a1': 0.46 * scale}))
for scale in [0.8, 1.0, 1.2]:
    hamming_variations.append((f'a0_{scale:.2f}', {'a0': 0.54 * scale, 'a1': 0.46}))

for label, params in hamming_variations:
    freqs, times, power = stft_power(
        x,
        fs,
        window_name='hamming',
        nperseg=nperseg,
        noverlap=int(0.5 * nperseg),
        fmin=4.0,
        fmax=100.0,
        window_kwargs=params,
    )
    title = (
        f'Chirp STFT (window=Hamming {label}, N={nperseg}, '
        f'overlap={int(0.5 * nperseg)} samples, fs={fs:.1f} Hz, fmin=4 Hz, fmax=100 Hz)'
    )
    save_path = FIG_DIR / f'chirp_hamming_{label}.png'
    plot_spectrogram(freqs, times, power, title=title, save_path=save_path)
    print(f'Saved: {save_path}')

Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_hamming_a1_0.80.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_hamming_a1_1.00.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_hamming_a1_1.20.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_hamming_a0_0.80.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_hamming_a0_1.00.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_hamming_a0_1.20.png


### 4.2 Blackman Sensitivity
We vary one coefficient at a time while holding the others at default.

In [36]:
# Blackman sensitivity
blackman_base = {'a0': 0.42, 'a1': 0.5, 'a2': 0.08}
blackman_variations = []
for scale in [0.8, 1.0, 1.2]:
    blackman_variations.append((f'a0_{scale:.2f}', {'a0': 0.42 * scale, 'a1': 0.5, 'a2': 0.08}))
    blackman_variations.append((f'a1_{scale:.2f}', {'a0': 0.42, 'a1': 0.5 * scale, 'a2': 0.08}))
    blackman_variations.append((f'a2_{scale:.2f}', {'a0': 0.42, 'a1': 0.5, 'a2': 0.08 * scale}))

for label, params in blackman_variations:
    freqs, times, power = stft_power(
        x,
        fs,
        window_name='blackman',
        nperseg=nperseg,
        noverlap=int(0.5 * nperseg),
        fmin=4.0,
        fmax=100.0,
        window_kwargs=params,
    )
    title = (
        f'Chirp STFT (window=Blackman {label}, N={nperseg}, '
        f'overlap={int(0.5 * nperseg)} samples, fs={fs:.1f} Hz, fmin=4 Hz, fmax=100 Hz)'
    )
    save_path = FIG_DIR / f'chirp_blackman_{label}.png'
    plot_spectrogram(freqs, times, power, title=title, save_path=save_path)
    print(f'Saved: {save_path}')

Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_blackman_a0_0.80.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_blackman_a1_0.80.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_blackman_a2_0.80.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_blackman_a0_1.00.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_blackman_a1_1.00.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_blackman_a2_1.00.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_blackman_a0_1.20.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_blackman_a1_1.20.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_blackman_a2_1.20.png


### 4.3 Gaussian Sensitivity
We sweep $\sigma$ from 0.01 to 0.5.

In [37]:
# Gaussian sensitivity
sigmas = np.linspace(0.01, 0.5, 10)
for sigma in sigmas:
    freqs, times, power = stft_power(
        x,
        fs,
        window_name='gaussian',
        nperseg=nperseg,
        noverlap=int(0.5 * nperseg),
        fmin=4.0,
        fmax=100.0,
        window_kwargs={'sigma': float(sigma)},
    )
    title = (
        f'Chirp STFT (window=Gaussian sigma={sigma:.2f}, N={nperseg}, '
        f'overlap={int(0.5 * nperseg)} samples, fs={fs:.1f} Hz, fmin=4 Hz, fmax=100 Hz)'
    )
    save_path = FIG_DIR / f'chirp_gaussian_{sigma:.2f}.png'
    plot_spectrogram(freqs, times, power, title=title, save_path=save_path)
    print(f'Saved: {save_path}')

Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_gaussian_0.01.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_gaussian_0.06.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_gaussian_0.12.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_gaussian_0.17.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_gaussian_0.23.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_gaussian_0.28.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_gaussian_0.34.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_gaussian_0.39.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_gaussian_0.45.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/chirp_gaussian_0.50.png


## 5. EEG Data (First Channel)

We load the first channel from the EDF file and compute a baseline spectrogram.

In [38]:
eeg_path = PROJECT_ROOT / 'data' / 'sub-NORB00055_ses-1_task-EEG_eeg.edf'
eeg_signal, fs_eeg, channel_name = load_first_channel_edf(eeg_path)
print(f'Loaded channel: {channel_name}, fs={fs_eeg:.2f} Hz, samples={eeg_signal.size}')

# Limit to a shorter segment to keep sensitivity sweeps tractable
eeg_seconds = 10.0
eeg_samples = int(fs_eeg * eeg_seconds)
eeg_signal = eeg_signal[: min(eeg_samples, eeg_signal.size)]
print(f'Using {eeg_signal.size} samples for EEG analysis')

# Plot the original EEG waveform
t_eeg = np.arange(eeg_signal.size) / fs_eeg
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(t_eeg, eeg_signal, color='tab:blue', linewidth=0.8)
ax.set_title(
    f'EEG waveform (channel={channel_name}, fs={fs_eeg:.1f} Hz, duration={eeg_signal.size / fs_eeg:.1f} s)'
)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.grid(alpha=0.3)
fig.tight_layout()
save_path = FIG_DIR / 'eeg_waveform.png'
fig.savefig(save_path, dpi=180)
plt.close(fig)
print(f'Saved: {save_path}')

nperseg_eeg = compute_window_length(fs_eeg, f_min=4.0, cycles=3)
noverlap_eeg = int(0.5 * nperseg_eeg)
freqs, times, power = stft_power(
    eeg_signal,
    fs_eeg,
    window_name='hann',
    nperseg=nperseg_eeg,
    noverlap=noverlap_eeg,
    fmin=4.0,
    fmax=100.0,
)

title = (
    f'EEG STFT (window=Hann, channel={channel_name}, N={nperseg_eeg}, '
    f'overlap={noverlap_eeg} samples, fs={fs_eeg:.1f} Hz, fmin=4 Hz, fmax=100 Hz)'
)
save_path = FIG_DIR / 'eeg_hann_baseline.png'
plot_spectrogram(freqs, times, power, title=title, save_path=save_path)
print(f'Saved: {save_path}')

Loaded channel: Fp1, fs=200.00 Hz, samples=228000
Using 2000 samples for EEG analysis
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_waveform.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_hann_baseline.png


### 5.1 EEG Overlap Sweep

We repeat the overlap sweep on the EEG segment to compare time sampling density.

In [39]:
# EEG overlap sweep
eeg_overlap_fracs = [i / 100 for i in range(0, 100, 5) if i < 95] + [0.95]
eeg_overlap_results = overlap_sweep(
    eeg_signal,
    fs_eeg,
    window_name='hann',
    nperseg=nperseg_eeg,
    overlap_fracs=eeg_overlap_fracs,
    fmin=4.0,
    fmax=100.0,
)

for frac in [0.0, 0.5, 0.9, 0.95]:
    freqs, times, power = eeg_overlap_results[frac]
    title = (
        f'EEG STFT (window=Hann, overlap={int(frac*100)}%, channel={channel_name}, '
        f'N={nperseg_eeg}, fs={fs_eeg:.1f} Hz, fmin=4 Hz, fmax=100 Hz)'
    )
    save_path = FIG_DIR / f'eeg_hann_overlap_{int(frac*100)}.png'
    plot_spectrogram(freqs, times, power, title=title, save_path=save_path)
    print(f'Saved: {save_path}')

Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_hann_overlap_0.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_hann_overlap_50.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_hann_overlap_90.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_hann_overlap_95.png


### 5.2 EEG Hamming Sensitivity

We vary Hamming coefficients on the EEG segment using the same +/-20% sweep.

In [40]:
# EEG Hamming sensitivity
eeg_hamming_variations = []
for scale in [0.8, 1.0, 1.2]:
    eeg_hamming_variations.append((f'a1_{scale:.2f}', {'a0': 0.54, 'a1': 0.46 * scale}))
for scale in [0.8, 1.0, 1.2]:
    eeg_hamming_variations.append((f'a0_{scale:.2f}', {'a0': 0.54 * scale, 'a1': 0.46}))

for label, params in eeg_hamming_variations:
    freqs, times, power = stft_power(
        eeg_signal,
        fs_eeg,
        window_name='hamming',
        nperseg=nperseg_eeg,
        noverlap=int(0.5 * nperseg_eeg),
        fmin=4.0,
        fmax=100.0,
        window_kwargs=params,
    )
    title = (
        f'EEG STFT (window=Hamming {label}, channel={channel_name}, N={nperseg_eeg}, '
        f'overlap={int(0.5 * nperseg_eeg)} samples, fs={fs_eeg:.1f} Hz, fmin=4 Hz, fmax=100 Hz)'
    )
    save_path = FIG_DIR / f'eeg_hamming_{label}.png'
    plot_spectrogram(freqs, times, power, title=title, save_path=save_path)
    print(f'Saved: {save_path}')

Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_hamming_a1_0.80.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_hamming_a1_1.00.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_hamming_a1_1.20.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_hamming_a0_0.80.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_hamming_a0_1.00.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_hamming_a0_1.20.png


### 5.3 EEG Blackman Sensitivity

We vary one Blackman coefficient at a time on the EEG segment.

In [41]:
# EEG Blackman sensitivity
eeg_blackman_variations = []
for scale in [0.8, 1.0, 1.2]:
    eeg_blackman_variations.append((f'a0_{scale:.2f}', {'a0': 0.42 * scale, 'a1': 0.5, 'a2': 0.08}))
    eeg_blackman_variations.append((f'a1_{scale:.2f}', {'a0': 0.42, 'a1': 0.5 * scale, 'a2': 0.08}))
    eeg_blackman_variations.append((f'a2_{scale:.2f}', {'a0': 0.42, 'a1': 0.5, 'a2': 0.08 * scale}))

for label, params in eeg_blackman_variations:
    freqs, times, power = stft_power(
        eeg_signal,
        fs_eeg,
        window_name='blackman',
        nperseg=nperseg_eeg,
        noverlap=int(0.5 * nperseg_eeg),
        fmin=4.0,
        fmax=100.0,
        window_kwargs=params,
    )
    title = (
        f'EEG STFT (window=Blackman {label}, channel={channel_name}, N={nperseg_eeg}, '
        f'overlap={int(0.5 * nperseg_eeg)} samples, fs={fs_eeg:.1f} Hz, fmin=4 Hz, fmax=100 Hz)'
    )
    save_path = FIG_DIR / f'eeg_blackman_{label}.png'
    plot_spectrogram(freqs, times, power, title=title, save_path=save_path)
    print(f'Saved: {save_path}')

Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_blackman_a0_0.80.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_blackman_a1_0.80.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_blackman_a2_0.80.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_blackman_a0_1.00.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_blackman_a1_1.00.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_blackman_a2_1.00.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_blackman_a0_1.20.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_blackman_a1_1.20.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_blackman_a2_1.20.png


### 5.4 EEG Gaussian Sensitivity

We sweep $\sigma$ from 0.01 to 0.5 on the EEG segment.

In [42]:
# EEG Gaussian sensitivity
eeg_sigmas = np.linspace(0.01, 0.5, 10)
for sigma in eeg_sigmas:
    freqs, times, power = stft_power(
        eeg_signal,
        fs_eeg,
        window_name='gaussian',
        nperseg=nperseg_eeg,
        noverlap=int(0.5 * nperseg_eeg),
        fmin=4.0,
        fmax=100.0,
        window_kwargs={'sigma': float(sigma)},
    )
    title = (
        f'EEG STFT (window=Gaussian sigma={sigma:.2f}, channel={channel_name}, N={nperseg_eeg}, '
        f'overlap={int(0.5 * nperseg_eeg)} samples, fs={fs_eeg:.1f} Hz, fmin=4 Hz, fmax=100 Hz)'
    )
    save_path = FIG_DIR / f'eeg_gaussian_{sigma:.2f}.png'
    plot_spectrogram(freqs, times, power, title=title, save_path=save_path)
    print(f'Saved: {save_path}')

Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_gaussian_0.01.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_gaussian_0.06.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_gaussian_0.12.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_gaussian_0.17.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_gaussian_0.23.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_gaussian_0.28.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_gaussian_0.34.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_gaussian_0.39.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_gaussian_0.45.png
Saved: /home/plutonium/Documents/Biosignals/project/notebooks/week2/figures/eeg_gaussian_0.50.png


## 6. Discussion

- Hamming/Blackman coefficient changes alter sidelobe levels and main-lobe width,
  trading spectral leakage for resolution.
- Gaussian $\sigma$ controls time-frequency localization: smaller $\sigma$ narrows the window,
  improving time resolution but broadening spectral peaks.
- Overlap changes temporal sampling density of frames, affecting smoothness of the spectrogram.